# Exploratory Alarm-Onset Evidence Score

This notebook recalculates the exploratory alarm-onset evidence score for every available site and parameter, selects the strongest eligible parameter for each site, and generates the cross-site comparison graph.

The score is an exploratory ranking measure, not a probability, causal estimate, statistical significance test, or manufacturer-defined alarm threshold.


This mirror reads only `site_alarm_correlation_results` under the new isolated output root.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_workspace():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository results directory")


WORKSPACE = find_workspace()
RESULTS_DIR = WORKSPACE / "results"
print("Results directory:", RESULTS_DIR)


## Formula

For parameter \(x\), the evidence score is:

\[
E = \min\left(\left|\frac{\tilde{x}_{alarm}-\tilde{x}_{baseline}}{IQR_{baseline}}\right|,5\right)
\times \ln(1+n)
\times \max(P_{low},P_{high})
\]

- \(\tilde{x}_{alarm}\): median value at mapped alarm starts.
- \(\tilde{x}_{baseline}\): median during non-alarm periods.
- \(IQR_{baseline}\): baseline interquartile range.
- \(n\): mapped alarm-start observations.
- \(P_{low}\): fraction below the baseline 10th percentile.
- \(P_{high}\): fraction above the baseline 90th percentile.

Candidate scores are promoted only when at least 10 alarm-start values are mapped and a high/low candidate threshold is available.


In [ ]:
finding_frames = []
for site_dir in sorted(RESULTS_DIR.glob("site_*")):
    finding_path = site_dir / "parameter_alarm_onset_findings.csv"
    if not finding_path.exists():
        continue
    findings = pd.read_csv(finding_path)
    findings.insert(0, "site_id", int(site_dir.name.split("_")[1]))
    finding_frames.append(findings)

onset_findings = pd.concat(finding_frames, ignore_index=True, sort=False)

capped_shift = onset_findings["standardized_event_shift_iqr"].abs().fillna(0).clip(upper=5)
extreme_fraction = (
    onset_findings[
        [
            "alarm_values_below_baseline_p10_pct",
            "alarm_values_above_baseline_p90_pct",
        ]
    ]
    .max(axis=1)
    .fillna(0)
    / 100
)

onset_findings["recalculated_evidence_score"] = (
    capped_shift
    * np.log1p(onset_findings["mapped_alarm_values"])
    * extreme_fraction
)
onset_findings.loc[
    (onset_findings["mapped_alarm_values"] < 10)
    | onset_findings["candidate_threshold"].isna(),
    "recalculated_evidence_score",
] = 0

eligible = onset_findings[onset_findings["recalculated_evidence_score"] > 0].copy()
strongest_by_site = (
    eligible.sort_values("recalculated_evidence_score", ascending=False)
    .groupby("site_id", as_index=False)
    .head(1)
    .sort_values("recalculated_evidence_score")
)

display(
    strongest_by_site[
        [
            "site_id",
            "parameter",
            "mapped_alarm_values",
            "candidate_direction",
            "candidate_threshold",
            "recalculated_evidence_score",
        ]
    ]
)


In [ ]:
labels = [
    f"{int(row.site_id)}: {row.parameter}"
    for row in strongest_by_site.itertuples(index=False)
]
colors = [
    "#dc2626" if "high values" in str(direction) else "#2563eb"
    for direction in strongest_by_site["candidate_direction"]
]

fig, axis = plt.subplots(figsize=(13, max(6, len(strongest_by_site) * 0.55)))
axis.barh(labels, strongest_by_site["recalculated_evidence_score"], color=colors)
axis.set_xlabel("Exploratory onset evidence score")
axis.set_title("Strongest parameter pattern at exact alarm starts by site")
axis.grid(True, axis="x", alpha=0.2)
fig.tight_layout()

output_path = RESULTS_DIR / "all_sites_top_alarm_onset_candidates.png"
fig.savefig(output_path, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Saved:", output_path)
